# Machine Learning with Tabular Data: Diabetes Readmission Prediction

## Objective and Scope

The objective is to predict the readmission status of an inpatient encounter
with a recorded diabetes diagnosis using three classes:

- `NO`: no subsequent inpatient readmission is recorded within the available data.
- `>30`: readmission occurs more than 30 days after discharge.
- `<30`: readmission occurs less than 30 days after discharge.

The prediction time is defined as **the time of discharge**. Information collected
during the hospital stay may therefore be considered if it is available when the
prediction is made. The dataset does not provide detailed recording timestamps for
individual features, so this availability remains an explicit deployment assumption.

The unit of prediction is an inpatient encounter. A patient may have multiple
encounters, but the same patient must not appear in both the training and test sets.

The primary cohort excludes encounters ending in death or discharge to hospice,
using the corresponding codes documented in `IDS_mapping.csv`. This defines the
study population; it is not a claim that hospice patients can never be readmitted.

The numerical target labels 0, 1, and 2 are class identifiers. They do not represent
numerical distance or an ordering of clinical severity.

## Workflow

1. Initial data inspection and data-quality assessment.
2. Cohort definition and patient-level train-test split.
3. Detailed EDA using the training set only.
4. Deterministic feature engineering and train-fitted preprocessing.
5. Comparison of representations with and without PCA.
6. Group-aware cross-validation and final evaluation on the held-out test set.


# Phase 1: Data Acquisition and Initial Inspection

## 1.1 Download and Extract the Dataset

In [ ]:
import os
import sys
import subprocess
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Root-level project setup for local execution and Google Colab.
REPO_URL = "https://github.com/nam21-12-06/machine-learning-assignment.git"
REPO_BRANCH = "features/model_svm"

if "google.colab" in sys.modules:
    print("Running on Google Colab")
    PROJECT_DIR = Path("/content/machine-learning-assignment")
    if not PROJECT_DIR.exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_DIR),
            ],
            check=True,
        )
else:
    print("Running on Local Machine")
    PROJECT_DIR = Path.cwd().resolve()

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
MODULES_DIR = PROJECT_DIR / "modules"
REPORTS_DIR = PROJECT_DIR / "reports"
FEATURES_DIR = PROJECT_DIR / "features"
MODELS_DIR = PROJECT_DIR / "models"

for directory in [DATA_DIR, RAW_DIR, REPORTS_DIR, FEATURES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

required_modules = [
    MODULES_DIR / "preprocessing.py",
    MODULES_DIR / "svm_model.py",
    MODULES_DIR / "evaluation.py",
    MODULES_DIR / "logistic_regression_model.py",
    MODULES_DIR / "random_forest_model.py",
]
missing_modules = [path.name for path in required_modules if not path.exists()]
if missing_modules:
    raise FileNotFoundError(
        "Required project modules are missing: " + ", ".join(missing_modules)
    )

print(f"Project directory set to: {PROJECT_DIR}")
print("Supporting modules and output directories are ready.")

In [ ]:
# Dowload dataset
DATA_URL = "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"

ZIP_PATH = DATA_DIR / "diabetes.zip"

if not ZIP_PATH.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print("Download completed.")
else:
    print("Dataset archive already exists. Skipping download.")

In [ ]:
# Extract dataset
if not list(RAW_DIR.glob("*.csv")):
    print("Extracting files...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(RAW_DIR)
    print("Extraction completed.")
else:
    print("Files already extracted.")
print("\nFiles in raw directory:")
for path in RAW_DIR.rglob("*"):
    if path.is_file():
        print(f" - {path.name}")

## 1.2 Load the Dataset

Identifiers and diagnosis codes are loaded as strings to preserve their meaning.

Missing-value parsing is explicitly controlled. In laboratory features, the
literal value `None` represents a test that was not performed and is renamed
`NotMeasured`. It is kept separate from genuinely missing information.

The original loaded data are retained in `df_raw`, while `df` is used for
deterministic standardization and initial inspection.

In [ ]:
TARGET = "readmitted"
CLASS_ORDER = ["NO", ">30", "<30"]
LAB_COLS = ["max_glu_serum", "A1Cresult"]
IDENTIFIER_COLS = ["encounter_id", "patient_nbr"]
CODE_COLS = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
]
DIAG_COLS = ["diag_1", "diag_2", "diag_3"]

data_paths = list(RAW_DIR.rglob("diabetic_data.csv"))
assert len(data_paths) == 1, "Expected exactly one diabetic_data.csv file."
DATA_PATH = data_paths[0]

# encounter_id remains numeric. The other identifiers/codes are categorical strings.
categorical_string_cols = ["patient_nbr"] + CODE_COLS + DIAG_COLS
df_raw = pd.read_csv(
    DATA_PATH,
    keep_default_na=False,
    na_values=["?", ""],
    dtype={column: object for column in categorical_string_cols},
    low_memory=False,
)

df = df_raw.copy()

# Strip whitespace without converting genuine NaN values into the string "nan".
for column in df.select_dtypes(include="object"):
    df[column] = df[column].str.strip()

# "None" means the test was not ordered/performed, not an unknown data value.
for column in LAB_COLS:
    df[column] = df[column].replace({"None": "NotMeasured"})

assert df[TARGET].notna().all(), "The target contains missing values."
assert set(df[TARGET].unique()) <= set(CLASS_ORDER), "Unexpected target class found."
assert df[IDENTIFIER_COLS].notna().all().all(), "An identifier is missing."

print("Dataset shape:", df.shape)
display(df[LAB_COLS].head())


In [ ]:
print(f"Dataset shape:{df.shape}") # 50 features and 101766 samples
print(f"Number of rows:{df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
# Columns names
print("COLUMN NAMES")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2d}. {col}")

In [ ]:
# Print first 5 rows
display(df.head())

In [ ]:
# Data type
display(df.dtypes.to_frame(name="dtype"))

## 1.3 Initial Data Quality Assessment

This section describes the structure and basic quality of the complete dataset.

It distinguishes:

- Missing information from the NotMeasured laboratory state.
- Fully duplicated rows from repeated encounters belonging to the same patient.
- Identifier columns from predictive features.

A repeated patient identifier does not automatically indicate a duplicate record.
It indicates that the patient has multiple recorded encounters.

In [ ]:
print("=" * 60)
print("DATASET OVERVIEW - NUMERICAL & CATEGORICAL")
print("=" * 60)

# Keep encounter_id numeric so chronological sorting remains numeric later.
# Categorical ID columns are converted to strings to avoid treating them as measurements.
id_cols = ['patient_nbr', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id']
df[id_cols] = df[id_cols].astype(str)

# Observe overall distribution
display(df.describe(include="all").T)

## 1.4 Missing Value Analysis

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
}).sort_values("missing_pct", ascending=False)

print("Missing information:")
display(missing_summary.query("missing_count > 0").round(2))

lab_status_summary = pd.DataFrame({
    "not_measured_count": [
        df[col].eq("NotMeasured").sum() for col in LAB_COLS
    ],
    "not_measured_pct": [
        df[col].eq("NotMeasured").mean() * 100 for col in LAB_COLS
    ],
    "missing_count": [
        df[col].isna().sum() for col in LAB_COLS
    ],
}, index=LAB_COLS)

print("Laboratory test availability:")
display(lab_status_summary.round(2))

visits_per_patient = df.groupby("patient_nbr").size()

print("Fully duplicated rows:", df.duplicated().sum())
print("Repeated encounter IDs:", df["encounter_id"].duplicated().sum())
print("Unique patients:", visits_per_patient.size)
print("Patients with multiple encounters:", visits_per_patient.gt(1).sum())
print("Encounters beyond one per patient:", (visits_per_patient - 1).sum())

display(visits_per_patient.describe().to_frame("encounters_per_patient"))

## 1.5 Duplicate Analysis

In [ ]:
print("=" * 60)
print("DUPLICATE & PATIENT ENCOUNTER ANALYSIS")
print("=" * 60)

# Check for completely duplicated rows
duplicate_count = df.duplicated().sum()
print(f"Total fully duplicated rows: {duplicate_count}")

# BONUS: Check if a single patient has multiple encounters (readmissions)
unique_patients = df['patient_nbr'].nunique()
total_encounters = len(df)
multiple_encounters = total_encounters - unique_patients

print(f"Total encounters (rows): {total_encounters:,}")
print(f"Total unique patients: {unique_patients:,}")
print(f"Number of return visits (multiple encounters): {multiple_encounters:,}")


## 1.6 Target Distribution

In [ ]:
TARGET = "readmitted"

# Count each target class
target_counts = df[TARGET].value_counts()

# Calculate percentages
target_percentages = df[TARGET].value_counts(normalize=True) * 100

# Combine into one summary table
target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

print("=" * 60)
print("TARGET DISTRIBUTION")
print("=" * 60)

display(target_summary)

Target Distribution Visualization

In [ ]:
# Visualization
plt.figure(figsize=(9, 6))
# Use 'viridis' palette for better aesthetic, assign x to hue to avoid warnings

ax = sns.countplot(data=df, x=TARGET, order=target_counts.index, hue=TARGET, palette='viridis', legend=False)
plt.title("Distribution of Readmission Classes (Imbalanced)", fontsize=14, fontweight='bold')
plt.xlabel("Readmission Status", fontsize=12)
plt.ylabel("Number of Encounters", fontsize=12)

# Add both count and percentage labels on top of the bars
for i, container in enumerate(ax.containers):
    labels = [f'{v.get_height():,} \n({target_percentages.iloc[j]:.1f}%)' for j, v in enumerate(container)]
    ax.bar_label(container, labels=labels, padding=5, fontweight='bold')

# Expand y-axis slightly to fit the tall labels
plt.ylim(0, target_counts.max() * 1.2)
plt.tight_layout()
plt.show()

# Phase 2: Cohort Definition and Patient-Level Holdout Split

The primary analysis excludes encounters ending in death or discharge to hospice,
according to the supplied ID mapping. All other eligible encounters are retained.

The holdout split is performed by `patient_nbr` before detailed EDA. Several
Group Shuffle Split candidates are generated, and the candidate closest to the
overall class distribution is selected. This approximates stratification while
preserving complete patient separation.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

DEAD_HOSPICE_IDS = {"11", "13", "14", "19", "20", "21"}

df_cohort = df.loc[
    ~df["discharge_disposition_id"].isin(DEAD_HOSPICE_IDS)
].copy()

print("Original encounters:", f"{len(df):,}")
print("Eligible encounters:", f"{len(df_cohort):,}")
print("Excluded death/hospice encounters:", f"{len(df) - len(df_cohort):,}")

overall_distribution = (
    df_cohort[TARGET].value_counts(normalize=True).reindex(CLASS_ORDER, fill_value=0)
)

splitter = GroupShuffleSplit(n_splits=20, test_size=0.20, random_state=RANDOM_STATE)
split_candidates = []
for candidate_id, (candidate_train, candidate_test) in enumerate(
    splitter.split(df_cohort, df_cohort[TARGET], groups=df_cohort["patient_nbr"])
):
    test_distribution = (
        df_cohort.iloc[candidate_test][TARGET]
        .value_counts(normalize=True)
        .reindex(CLASS_ORDER, fill_value=0)
    )
    class_gap = (test_distribution - overall_distribution).abs().sum()
    size_gap = abs(len(candidate_test) / len(df_cohort) - 0.20)
    split_candidates.append((class_gap + size_gap, candidate_id, candidate_train, candidate_test))

_, selected_candidate, train_positions, test_positions = min(
    split_candidates, key=lambda item: item[0]
)

df_train_raw = df_cohort.iloc[train_positions].copy()
df_test_raw = df_cohort.iloc[test_positions].copy()

train_patients = set(df_train_raw["patient_nbr"])
test_patients = set(df_test_raw["patient_nbr"])
patient_overlap = train_patients.intersection(test_patients)
assert not patient_overlap, "Patient leakage detected between train and test."

split_distribution = pd.concat({
    "cohort_pct": overall_distribution * 100,
    "train_pct": df_train_raw[TARGET].value_counts(normalize=True).reindex(CLASS_ORDER) * 100,
    "test_pct": df_test_raw[TARGET].value_counts(normalize=True).reindex(CLASS_ORDER) * 100,
}, axis=1)

print("Selected GroupShuffleSplit candidate:", selected_candidate)
print("Train encounters / patients:", f"{len(df_train_raw):,}", f"/ {len(train_patients):,}")
print("Test encounters / patients:", f"{len(df_test_raw):,}", f"/ {len(test_patients):,}")
print("Patient overlap:", len(patient_overlap))
display(split_distribution.round(2))

# All detailed EDA below uses only the training partition.
eda_df = df_train_raw


# Phase 3: Exploratory Data Analysis on the Training Set

## 3.1 Categorical Features


In [ ]:
TARGET = "readmitted"

print("=" * 60)
print("TRAINING-SET CATEGORICAL SUMMARY")
print("=" * 60)

categorical_cols = eda_df.select_dtypes(include=["object", "string"]).columns.tolist()
excluded_cols = [TARGET, "encounter_id", "patient_nbr"]
categorical_features_eda = [c for c in categorical_cols if c not in excluded_cols]

cat_summary_rows = []
for column in categorical_features_eda:
    values = eda_df[column].fillna("Missing")
    counts = values.value_counts()
    cat_summary_rows.append({
        "feature": column,
        "unique_values": values.nunique(),
        "dominant_value": counts.index[0],
        "dominant_pct": counts.iloc[0] / len(eda_df) * 100,
    })

cat_summary = pd.DataFrame(cat_summary_rows).sort_values("dominant_pct", ascending=False)
display(cat_summary.round(2))

near_constant_eda = cat_summary.query("dominant_pct > 95")
print(f"Near-constant candidates (>95% dominant): {len(near_constant_eda)}")
print("These are candidates for validation, not automatically useless features.")


In [ ]:
print("=" * 60)
print("DISTRIBUTION OF KEY CATEGORICAL FEATURES - TRAINING SET")
print("=" * 60)

key_categorical_features = ["gender", "race", "age", "change", "diabetesMed", "insulin"]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for axis, column in zip(axes, key_categorical_features):
    plot_data = eda_df[column].fillna("Missing").astype(str)
    sns.countplot(
        x=plot_data,
        order=plot_data.value_counts().index,
        hue=plot_data,
        palette="Set2",
        legend=False,
        ax=axis,
    )
    axis.set_title(f"Distribution of {column}", fontsize=13, fontweight="bold")
    axis.set_xlabel("")
    axis.set_ylabel("Number of encounters")
    axis.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## 3.2 Numerical Features and Outliers


In [ ]:
print("=" * 60)
print("NUMERICAL DISTRIBUTIONS AND OUTLIERS - TRAINING SET")
print("=" * 60)

numerical_cols = [
    column for column in eda_df.select_dtypes(include=["int64", "float64"]).columns
    if column != "encounter_id"
]

numeric_summary_rows = []
for column in numerical_cols:
    values = eda_df[column].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    outlier_mask = (values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)
    numeric_summary_rows.append({
        "feature": column,
        "skewness": values.skew(),
        "iqr_outlier_pct": outlier_mask.mean() * 100,
        "median": values.median(),
    })

numeric_summary = pd.DataFrame(numeric_summary_rows).sort_values(
    "skewness", key=lambda series: series.abs(), ascending=False
)
display(numeric_summary.round(2))

fig, axes = plt.subplots(len(numerical_cols), 2, figsize=(14, 4 * len(numerical_cols)))
for row, column in enumerate(numerical_cols):
    sns.histplot(eda_df[column], kde=True, ax=axes[row, 0], color="steelblue")
    axes[row, 0].set_title(f"Distribution of {column}", fontweight="bold")
    sns.boxplot(x=eda_df[column], ax=axes[row, 1], color="mediumseagreen")
    axes[row, 1].set_title(f"Boxplot of {column}", fontweight="bold")

plt.tight_layout()
plt.show()


## 3.3 Numerical Correlation


In [ ]:
print("=" * 60)
print("NUMERICAL CORRELATION - TRAINING SET")
print("=" * 60)

corr_matrix = eda_df[numerical_cols].corr()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
top_correlations = (
    upper_triangle.stack()
    .rename("correlation")
    .to_frame()
    .assign(abs_correlation=lambda table: table["correlation"].abs())
    .sort_values("abs_correlation", ascending=False)
    .head(10)
)

print("Ten strongest pairwise correlations by absolute value:")
display(top_correlations.round(3))

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
)
plt.title("Correlation Heatmap of Numerical Features (Training Set)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
print("=" * 60)
print("BIVARIATE ANALYSIS - TRAINING SET")
print("=" * 60)

time_summary = eda_df.groupby(TARGET, observed=False)["time_in_hospital"].agg(
    count="size", mean="mean", median="median", std="std"
).reindex(CLASS_ORDER)
print("Time in hospital by target class:")
display(time_summary.round(2))

def target_rate_table(data, feature):
    values = data[feature].fillna("Missing")
    table = pd.crosstab(values, data[TARGET], normalize="index")
    return (table.reindex(columns=CLASS_ORDER, fill_value=0) * 100).round(2)

for feature in ["age", "gender", "race", "insulin", "A1Cresult"]:
    print(f"Target percentages within each {feature} group:")
    display(target_rate_table(eda_df, feature))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(data=eda_df, x=TARGET, y="time_in_hospital", order=CLASS_ORDER,
            color="mediumseagreen", ax=axes[0])
axes[0].set_title("Time in Hospital vs Readmission Status")

sns.countplot(data=eda_df, x="age", hue=TARGET, hue_order=CLASS_ORDER,
              palette="viridis", ax=axes[1])
axes[1].set_title("Age Group vs Readmission Status")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## 3.5 EDA Summary and Decisions

### Finding 1 - The target is meaningfully, but not extremely, imbalanced

`<30` is the smallest and most operationally important class. Class proportions are
reported for the full cohort and both patient-disjoint splits.

**Interpretation:** Accuracy alone can hide poor detection of early readmission.

**Action:** Report macro-F1, weighted-F1, the confusion matrix, and precision/recall/F1
for `<30`. Multiclass ROC-AUC or PR-AUC must state the One-vs-Rest and averaging scheme.

### Finding 2 - Repeated patients create a leakage risk, not automatic leakage

Multiple encounters from one patient are valid observations. Leakage occurs if the
same patient is present in both training and test data.

**Action:** Retain all eligible encounters and split by `patient_nbr`. Keep patient IDs
only for grouping; never expose them to the model. Use group-aware cross-validation.

### Finding 3 - Missingness and non-measurement are not always equivalent

`weight` has very little observed information. `max_glu_serum` and `A1Cresult` are often
not measured, which may describe the care process rather than a recording error.

**Action:** Drop `weight`, preserve `NotMeasured` as an explicit laboratory state, and
impute genuinely missing categorical values inside the training pipeline.

### Finding 4 - Low variance does not prove that a feature has no signal

Some medication columns are constant or nearly constant. Rare values can still be
associated with the rare `<30` outcome.

**Action:** Drop only zero-variance columns by default. Treat near-constant columns as
experiment candidates and compare model performance before removing them.

### Finding 5 - Prediction time determines whether a feature is valid

This notebook assumes prediction at discharge, so information available by discharge
may be used. In an admission-time deployment, fields such as
`discharge_disposition_id` would be future information and must be removed.

### Finding 6 - Visual counts must be separated from within-group risk

Large groups naturally produce more readmissions in count plots.

**Action:** Use within-group target percentages, sample sizes, numerical summaries,
and correlation tables alongside visualizations. Differences observed in EDA are
descriptive and are not evidence of causality.


# Phase 4: Data Preprocessing

The held-out test set was created before detailed EDA. Deterministic cleaning rules
are now applied consistently to the training and test frames. All data-dependent
operations, including imputation, rare-category grouping, scaling, feature selection,
and PCA, are fitted using training data only.


## 4.1 Initialize the Patient-Disjoint Splits


In [ ]:
print("=" * 60)
print("INITIALIZE CLEAN TRAIN AND TEST FRAMES")
print("=" * 60)

df_train_clean = df_train_raw.copy()
df_test_clean = df_test_raw.copy()

print(f"Training encounters: {len(df_train_clean):,}")
print(f"Test encounters: {len(df_test_clean):,}")
print("Repeated encounters are retained; patient groups remain disjoint.")


## 4.2 Remove Non-Predictive and Unusable Columns


In [ ]:
print("=" * 60)
print("DROP NON-PREDICTIVE AND HEAVILY MISSING COLUMNS")
print("=" * 60)

# patient_nbr is deliberately retained until X/y/groups are separated.
columns_to_drop = ["encounter_id", "weight"]
df_train_clean = df_train_clean.drop(columns=columns_to_drop, errors="ignore")
df_test_clean = df_test_clean.drop(columns=columns_to_drop, errors="ignore")

print("Dropped from both splits:", columns_to_drop)
print("patient_nbr retained temporarily for grouped validation.")
print("Train shape:", df_train_clean.shape)
print("Test shape:", df_test_clean.shape)


In [ ]:
print("=" * 60)
print("LOW-VARIANCE AUDIT ON TRAINING DATA")
print("=" * 60)

audit_features = [
    column for column in df_train_clean.columns
    if column not in {TARGET, "patient_nbr"}
]

variance_audit_rows = []
for column in audit_features:
    counts = df_train_clean[column].value_counts(dropna=False)
    variance_audit_rows.append({
        "feature": column,
        "unique_values": df_train_clean[column].nunique(dropna=False),
        "dominant_pct": counts.iloc[0] / len(df_train_clean) * 100,
    })

variance_audit = pd.DataFrame(variance_audit_rows).sort_values("dominant_pct", ascending=False)
constant_candidates = variance_audit.query("unique_values <= 1")["feature"].tolist()
near_constant_candidates = variance_audit.query(
    "unique_values > 1 and dominant_pct > 95"
)["feature"].tolist()

display(variance_audit.head(25).round(2))
print("Zero-variance columns to drop:", constant_candidates)
print("Near-constant columns retained for comparison:", near_constant_candidates)


In [ ]:
print("=" * 60)
print("CURRENT STATE BEFORE MISSING-CODE STANDARDIZATION")
print("=" * 60)

print(f"Training data: {df_train_clean.shape[0]:,} rows, {df_train_clean.shape[1]} columns")
print(f"Test data:     {df_test_clean.shape[0]:,} rows, {df_test_clean.shape[1]} columns")

train_missing = pd.DataFrame({
    "missing_count": df_train_clean.isna().sum(),
    "missing_pct": df_train_clean.isna().mean() * 100,
}).query("missing_count > 0").sort_values("missing_pct", ascending=False)

display(train_missing.round(2))


## 4.3 Standardize Hidden Missing Codes


In [ ]:
print("STANDARDIZING HIDDEN MISSING CODES")

invalid_code_map = {
    "admission_type_id": ["5", "6", "8"],
    "discharge_disposition_id": ["18", "25", "26"],
    "admission_source_id": ["9", "15", "17", "20", "21"],
}

def standardize_hidden_missing(frame):
    result = frame.copy()
    for column, invalid_codes in invalid_code_map.items():
        result[column] = result[column].replace(invalid_codes, np.nan)
    return result

df_train_clean = standardize_hidden_missing(df_train_clean)
df_test_clean = standardize_hidden_missing(df_test_clean)


In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df_train_clean.isna().sum(),
    "missing_pct": df_train_clean.isna().mean() * 100,
}).query("missing_count > 0").sort_values("missing_pct", ascending=False)

print("Training-set missing values after hidden-code standardization:")
display(missing_summary.round(2))

print("Laboratory states are retained separately from genuine missing values:")
display(pd.DataFrame({
    "not_measured_pct": [df_train_clean[c].eq("NotMeasured").mean() * 100 for c in LAB_COLS],
    "missing_pct": [df_train_clean[c].isna().mean() * 100 for c in LAB_COLS],
}, index=LAB_COLS).round(2))

print("No imputer is fitted here; imputation remains inside the sklearn pipeline.")


## 4.4 Feature Engineering: Age and ICD-9


In [ ]:
print("=" * 60)
print("CONFIGURABLE FEATURE ENGINEERING")
print("=" * 60)

AGE_ENCODING = "onehot"  # Alternative experiment: "midpoint"

age_mapping = {
    "[0-10)": 5, "[10-20)": 15, "[20-30)": 25, "[30-40)": 35,
    "[40-50)": 45, "[50-60)": 55, "[60-70)": 65, "[70-80)": 75,
    "[80-90)": 85, "[90-100)": 95,
}

def map_icd9_to_category(value):
    if pd.isna(value):
        return np.nan

    code = str(value).upper()
    if code.startswith("250"):
        return "Diabetes"
    if code.startswith(("V", "E")):
        return "Supplemental"

    try:
        number = float(code)
    except ValueError:
        return "Other"

    if 390 <= number <= 459 or number == 785:
        return "Circulatory"
    if 460 <= number <= 519 or number == 786:
        return "Respiratory"
    if 520 <= number <= 579 or number == 787:
        return "Digestive"
    if 800 <= number <= 999:
        return "Injury"
    if 710 <= number <= 739:
        return "Musculoskeletal"
    if 580 <= number <= 629 or number == 788:
        return "Genitourinary"
    if 140 <= number <= 239:
        return "Neoplasms"
    return "Other"

def apply_feature_engineering(frame, age_encoding=AGE_ENCODING):
    result = frame.copy()
    if age_encoding == "midpoint":
        result["age"] = result["age"].map(age_mapping)
    elif age_encoding != "onehot":
        raise ValueError("age_encoding must be 'onehot' or 'midpoint'")

    for column in DIAG_COLS:
        result[column] = result[column].map(map_icd9_to_category)
    return result

df_train_clean = apply_feature_engineering(df_train_clean)
df_test_clean = apply_feature_engineering(df_test_clean)

print("Age representation:", AGE_ENCODING)
print("ICD-9 codes mapped to 10 categories, including Supplemental and Other.")
display(df_train_clean["diag_1"].value_counts(dropna=False).to_frame("train_count"))


## 4.5 Target Encoding and Final Feature Matrices


In [ ]:
print("=" * 60)
print("TARGET ENCODING AND GROUP EXTRACTION")
print("=" * 60)

target_mapping = {"NO": 0, ">30": 1, "<30": 2}
target_names = {value: key for key, value in target_mapping.items()}

groups_train = df_train_clean.pop("patient_nbr")
groups_test = df_test_clean.pop("patient_nbr")

y_train = df_train_clean.pop(TARGET).map(target_mapping)
y_test = df_test_clean.pop(TARGET).map(target_mapping)
X_train = df_train_clean.copy()
X_test = df_test_clean.copy()

assert y_train.notna().all() and y_test.notna().all()
assert set(groups_train).isdisjoint(set(groups_test))

print("Target labels are class identifiers: NO=0, >30=1, <30=2")
print("Train shapes:", X_train.shape, y_train.shape)
print("Test shapes:", X_test.shape, y_test.shape)


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

print("=" * 60)
print("TRAIN-ONLY FEATURE FILTER AND GROUPED CROSS-VALIDATION")
print("=" * 60)

dominance = {
    column: X_train[column].value_counts(dropna=False).iloc[0] / len(X_train)
    for column in X_train.columns
}
constant_train_cols = [
    column for column in X_train.columns
    if X_train[column].nunique(dropna=False) <= 1
]
near_constant_train_cols = [
    column for column, ratio in dominance.items()
    if ratio > 0.95 and column not in constant_train_cols
]

# Default: remove only true constants. Near-constant features remain available
# for an explicit ablation experiment instead of being assumed useless.
DROP_NEAR_CONSTANT = False
columns_to_remove = constant_train_cols + (
    near_constant_train_cols if DROP_NEAR_CONSTANT else []
)

X_train = X_train.drop(columns=columns_to_remove)
X_test = X_test.drop(columns=columns_to_remove)

group_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Dropped zero-variance columns:", constant_train_cols)
print("Retained near-constant candidates:", near_constant_train_cols)
print("Group CV:", group_cv)
print("Final raw feature shapes:", X_train.shape, X_test.shape)


## 4.6 Configurable Preprocessing Pipeline


### 4.6.1 Numerical and Categorical Feature Groups


In [ ]:
print("=" * 60)
print("DEFINE FEATURE GROUPS")
print("=" * 60)

numerical_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

assert set(numerical_features).isdisjoint(categorical_features)
assert set(numerical_features + categorical_features) == set(X_train.columns)

print(f"Numerical features ({len(numerical_features)}):")
print(numerical_features)
print(f"Categorical features ({len(categorical_features)}):")
print(categorical_features)


In [ ]:
from modules.preprocessing import get_preprocessor

print("=" * 60)
print("BUILD CONFIGURABLE PREPROCESSOR")
print("=" * 60)

print("Available scaler options: standard, robust, minmax, none")
print("Rare categories are grouped using training-set frequency only.")


In [ ]:
print("=" * 60)
print("VALIDATE THE PREPROCESSOR")
print("=" * 60)

test_preprocessor = get_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scaler_type="standard",
    log_numeric=False,
    min_category_frequency=20,
)

X_train_processed = test_preprocessor.fit_transform(X_train)
X_test_processed = test_preprocessor.transform(X_test)

assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert np.isfinite(X_train_processed).all()
assert np.isfinite(X_test_processed).all()

print("Original train shape:", X_train.shape)
print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)
print("No non-finite values remain after preprocessing.")


# Phase 5: Feature Extraction and Dimensionality Reduction

PCA is treated as an experimental representation, not a mandatory preprocessing
step. Downstream models should compare no PCA, 90% explained variance, and 95%
explained variance under the same grouped cross-validation procedure.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

print("=" * 60)
print("PCA EXPLAINED VARIANCE - TRAINING DATA ONLY")
print("=" * 60)

preprocessor_pca = get_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scaler_type="standard",
    log_numeric=False,
    min_category_frequency=20,
)
X_train_pca_ready = preprocessor_pca.fit_transform(X_train)
X_test_pca_ready = preprocessor_pca.transform(X_test)

pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_train_pca_ready)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
threshold_90 = np.argmax(cumulative_variance >= 0.90) + 1
threshold_95 = np.argmax(cumulative_variance >= 0.95) + 1


In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance)
plt.axhline(0.90, color="red", linestyle="--", label=f"90% ({threshold_90} components)")
plt.axhline(0.95, color="green", linestyle="--", label=f"95% ({threshold_95} components)")
plt.axvline(threshold_90, color="red", linestyle="--")
plt.axvline(threshold_95, color="green", linestyle="--")
plt.title("PCA Cumulative Explained Variance - Training Set")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Encoded feature count:", X_train_pca_ready.shape[1])
print("Components for 90% variance:", threshold_90)
print("Components for 95% variance:", threshold_95)

def get_feature_pipeline(pca_variance=None, **preprocessor_options):
    """Create a comparable feature pipeline with optional train-fitted PCA."""
    steps = [("preprocessor", get_preprocessor(
        numerical_features=numerical_features,
        categorical_features=categorical_features,
        **preprocessor_options,
    ))]
    if pca_variance is not None:
        if pca_variance not in (0.90, 0.95):
            raise ValueError("pca_variance must be None, 0.90, or 0.95")
        steps.append(("pca", PCA(n_components=pca_variance, svd_solver="full")))
    return Pipeline(steps)

feature_pipeline_no_pca = get_feature_pipeline(pca_variance=None)
feature_pipeline_pca_90 = get_feature_pipeline(pca_variance=0.90)
feature_pipeline_pca_95 = get_feature_pipeline(pca_variance=0.95)

print("Feature pipelines prepared for downstream grouped model comparison.")


# Phase 6: Model Training and Evaluation

## 6.1: Support Vector Machine Training and Evaluation

This section trains and evaluates a Support Vector Machine classifier using the
patient-disjoint training and test sets prepared in the previous phases.

Model selection is performed exclusively on the training set using
StratifiedGroupKFold so that encounters from the same patient remain within the
same fold.

Because the dataset contains nearly 80,000 training encounters, LinearSVC is
used as the primary SVM implementation. This provides a computationally
practical maximum-margin classifier for the high-dimensional one-hot encoded
feature space.

The experiments compare:

- numerical scaling strategies;
- class weighting;
- PCA representations;
- regularization strength C.

Macro-F1 is used as the primary model-selection metric because the target is
imbalanced and the `<30` readmission class is the minority class.

The held-out test set is used only once after model selection.

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
    average_precision_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.dummy import DummyClassifier

from modules.evaluation import evaluate_classifier
from modules.svm_model import (
    build_linear_svm_pipeline,
    run_svm_cross_validation,
)

print("=" * 60)
print("SVM EXPERIMENT SETUP")
print("=" * 60)

LABELS = [0, 1, 2]
CLASS_NAMES = [target_names[label] for label in LABELS]

print("Class mapping:")
for label, name in zip(LABELS, CLASS_NAMES):
    print(f"  {label}: {name}")

print("\nPrimary selection metric: Macro-F1")
print("Cross-validation:", group_cv)

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
}

print("Evaluation metrics:")
for metric in scoring:
    print("-", metric)

In [ ]:
print("=" * 60)
print("DUMMY CLASSIFIER BASELINE")
print("=" * 60)

dummy_model = DummyClassifier(strategy="most_frequent")

dummy_cv = cross_validate(
    estimator=dummy_model,
    X=X_train,
    y=y_train,
    groups=groups_train,
    cv=group_cv,
    scoring=scoring,
    n_jobs=2,
)

dummy_summary = {
    metric: dummy_cv[f"test_{metric}"].mean()
    for metric in scoring
}

display(
    pd.DataFrame(
        [dummy_summary],
        index=["Dummy - Most Frequent"]
    ).round(4)
)

In [ ]:
print("SVM model and evaluation utilities imported successfully.")

In [ ]:
svm_configs = [
    {
        "name": "SVM-1 | Standard | No PCA | C=1 | No class weight",
        "scaler_type": "standard",
        "pca_variance": None,
        "C": 1.0,
        "class_weight": None,
    },
    {
        "name": "SVM-2 | Standard | No PCA | C=1 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "C": 1.0,
        "class_weight": "balanced",
    },
    {
        "name": "SVM-3 | MinMax | No PCA | C=1 | Balanced",
        "scaler_type": "minmax",
        "pca_variance": None,
        "C": 1.0,
        "class_weight": "balanced",
    },
    {
        "name": "SVM-4 | Standard | PCA 90% | C=1 | Balanced",
        "scaler_type": "standard",
        "pca_variance": 0.90,
        "C": 1.0,
        "class_weight": "balanced",
    },
    {
        "name": "SVM-5 | Standard | PCA 95% | C=1 | Balanced",
        "scaler_type": "standard",
        "pca_variance": 0.95,
        "C": 1.0,
        "class_weight": "balanced",
    },
    {
        "name": "SVM-6 | Standard | No PCA | C=0.1 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "C": 0.1,
        "class_weight": "balanced",
    },
    {
        "name": "SVM-7 | Standard | No PCA | C=10 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "C": 10.0,
        "class_weight": "balanced",
    },
]

print(f"Number of SVM configurations: {len(svm_configs)}")

In [ ]:
print("=" * 60)
print("RUN SVM CROSS-VALIDATION EXPERIMENTS")
print("=" * 60)

svm_results = run_svm_cross_validation(
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    group_cv=group_cv,
    configs=svm_configs,
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scoring=scoring,
    random_state=RANDOM_STATE,
    n_jobs=2,
)

display(svm_results.round(4))

In [ ]:
svm_ranking = svm_results[
    [
        "configuration",
        "cv_accuracy_mean",
        "cv_balanced_accuracy_mean",
        "cv_f1_macro_mean",
        "cv_f1_macro_std",
        "cv_f1_weighted_mean",
    ]
].copy()

display(svm_ranking.round(4))

plot_df = svm_results.sort_values(
    "cv_f1_macro_mean",
    ascending=True
)

ax = plot_df.plot(
    x="configuration",
    y="cv_f1_macro_mean",
    kind="barh",
    figsize=(10, 6),
    legend=False,
)

ax.set_xlabel("Mean Macro-F1")
ax.set_ylabel("Configuration")
ax.set_title("SVM Configuration Comparison - 5-Fold Grouped CV")

plt.tight_layout()
plt.show()

In [ ]:
best_config_name = svm_results.iloc[0]["configuration"]

config_lookup = {
    config["name"]: config
    for config in svm_configs
}

best_config = config_lookup[best_config_name]

print("=" * 60)
print("BEST SVM CONFIGURATION")
print("=" * 60)

print("Configuration:", best_config_name)
print("Scaler:", best_config["scaler_type"])
print("PCA:", best_config["pca_variance"])
print("C:", best_config["C"])
print("Class weight:", best_config["class_weight"])

print(
    "Mean CV Macro-F1:",
    round(svm_results.iloc[0]["cv_f1_macro_mean"], 4)
)

print(
    "Std CV Macro-F1:",
    round(svm_results.iloc[0]["cv_f1_macro_std"], 4)
)

In [ ]:
print("=" * 60)
print("FIT BEST SVM ON FULL TRAINING SET")
print("=" * 60)

best_svm = build_linear_svm_pipeline(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scaler_type=best_config["scaler_type"],
    pca_variance=best_config["pca_variance"],
    C=best_config["C"],
    class_weight=best_config["class_weight"],
    random_state=RANDOM_STATE,
)

best_svm.fit(
    X_train,
    y_train,
)

print("Best SVM fitted successfully.")

In [ ]:
print("=" * 60)
print("FINAL HELD-OUT TEST EVALUATION")
print("=" * 60)

y_test_pred = best_svm.predict(X_test)

In [ ]:
test_metrics, svm_test_summary, report_df = evaluate_classifier(
    y_true=y_test,
    y_pred=y_test_pred,
    labels=LABELS,
    class_names=CLASS_NAMES,
)

test_accuracy = test_metrics["Accuracy"]
test_balanced_accuracy = test_metrics["Balanced Accuracy"]
test_precision_macro = test_metrics["Macro Precision"]
test_recall_macro = test_metrics["Macro Recall"]
test_f1_macro = test_metrics["Macro F1"]
test_f1_weighted = test_metrics["Weighted F1"]

display(svm_test_summary.round(4))

In [ ]:
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

display(report_df.round(4))

In [ ]:
lt30_precision, lt30_recall, lt30_f1, lt30_support = (
    precision_recall_fscore_support(
        y_test,
        y_test_pred,
        labels=[2],
        zero_division=0,
    )
)

lt30_summary = pd.DataFrame(
    {
        "Metric": [
            "Precision",
            "Recall",
            "F1-score",
            "Support",
        ],
        "Value": [
            lt30_precision[0],
            lt30_recall[0],
            lt30_f1[0],
            lt30_support[0],
        ],
    }
)

display(lt30_summary.round(4))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    labels=LABELS,
    display_labels=CLASS_NAMES,
    normalize="true",
)

plt.title("Normalized SVM Confusion Matrix - Test Set")
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("ONE-VS-REST AVERAGE PRECISION")
print("=" * 60)

decision_scores = best_svm.decision_function(X_test)

y_test_binary = label_binarize(
    y_test,
    classes=LABELS,
)

macro_average_precision = average_precision_score(
    y_test_binary,
    decision_scores,
    average="macro",
)

lt30_average_precision = average_precision_score(
    (y_test == 2).astype(int),
    decision_scores[:, 2],
)

print(
    "Macro Average Precision (One-vs-Rest):",
    round(macro_average_precision, 4),
)

print(
    "<30 Average Precision (One-vs-Rest):",
    round(lt30_average_precision, 4),
)

In [ ]:
svm_final_result = pd.DataFrame(
    [
        {
            "Model": "Linear SVM",
            "Best Configuration": best_config_name,
            "Accuracy": test_accuracy,
            "Balanced Accuracy": test_balanced_accuracy,
            "Macro Precision": test_precision_macro,
            "Macro Recall": test_recall_macro,
            "Macro F1": test_f1_macro,
            "Weighted F1": test_f1_weighted,
            "<30 Precision": lt30_precision[0],
            "<30 Recall": lt30_recall[0],
            "<30 F1": lt30_f1[0],
            "Macro Average Precision OvR": macro_average_precision,
        }
    ]
)

display(svm_final_result.round(4))

In [ ]:
print("=" * 60)
print("EXPORT FINAL SVM FEATURES")
print("=" * 60)

feature_transformer = best_svm[:-1]
X_train_svm_features = np.asarray(
    feature_transformer.transform(X_train),
    dtype=np.float32,
)
X_test_svm_features = np.asarray(
    feature_transformer.transform(X_test),
    dtype=np.float32,
)

if "pca" in best_svm.named_steps:
    svm_feature_names = np.asarray(
        [
            f"PC{index:03d}"
            for index in range(1, X_train_svm_features.shape[1] + 1)
        ],
        dtype=str,
    )
else:
    svm_feature_names = np.asarray(
        best_svm.named_steps["preprocessor"].get_feature_names_out(),
        dtype=str,
    )

assert X_train_svm_features.shape[0] == len(y_train)
assert X_test_svm_features.shape[0] == len(y_test)
assert X_train_svm_features.shape[1] == X_test_svm_features.shape[1]
assert X_train_svm_features.shape[1] == len(svm_feature_names)
assert X_train_svm_features.dtype == np.float32
assert X_test_svm_features.dtype == np.float32

feature_artifacts = {
    "X_train_svm_features.npy": X_train_svm_features,
    "X_test_svm_features.npy": X_test_svm_features,
    "y_train.npy": np.asarray(y_train, dtype=np.int64),
    "y_test.npy": np.asarray(y_test, dtype=np.int64),
    "svm_feature_names.npy": svm_feature_names,
}

manifest_rows = []
for artifact_name, artifact_data in feature_artifacts.items():
    artifact_path = FEATURES_DIR / artifact_name
    np.save(artifact_path, artifact_data)
    manifest_rows.append(
        {
            "artifact": artifact_name,
            "shape": " x ".join(map(str, artifact_data.shape)),
            "dtype": str(artifact_data.dtype),
            "size_mb": artifact_path.stat().st_size / (1024 ** 2),
            "feature_count": X_train_svm_features.shape[1],
            "best_configuration": best_config_name,
        }
    )

svm_feature_manifest = pd.DataFrame(manifest_rows)
display(svm_feature_manifest.round({"size_mb": 2}))

SVM_REPORT_DIR = REPORTS_DIR / "svm"
SVM_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

svm_results.to_csv(
    SVM_REPORT_DIR / "svm_cv_results.csv",
    index=False,
)

svm_test_summary.to_csv(
    SVM_REPORT_DIR / "svm_test_metrics.csv",
    index=False,
)

svm_final_result.to_csv(
    SVM_REPORT_DIR / "svm_final_summary.csv",
    index=False,
)

report_df.to_csv(
    SVM_REPORT_DIR / "svm_classification_report.csv"
)

svm_feature_manifest.to_csv(
    SVM_REPORT_DIR / "svm_feature_manifest.csv",
    index=False,
)

print("Extracted feature files saved to:")
print(FEATURES_DIR)
print("SVM experiment results saved to:")
print(SVM_REPORT_DIR)

In [ ]:
import joblib

MODEL_PATH = MODELS_DIR / "best_linear_svm.joblib"

joblib.dump(
    best_svm,
    MODEL_PATH,
)

print("Best SVM pipeline saved to:")
print(MODEL_PATH)

### SVM Experimental Summary

A Linear Support Vector Machine was evaluated using patient-grouped
cross-validation on the training set. Macro-F1 was selected as the primary
model-selection metric because the target distribution is imbalanced and the
`<30` class represents the minority outcome.

The experiments investigated the effects of numerical scaling, class weighting,
PCA-based dimensionality reduction, and the SVM regularization parameter `C`.

The best configuration was selected exclusively from cross-validation results
and was subsequently refitted using the complete training set. The held-out
patient-disjoint test set was used only for the final evaluation.

The final analysis reports overall accuracy, balanced accuracy, macro-averaged
precision, recall and F1-score, weighted F1-score, class-specific performance,
and the confusion matrix. Particular attention is given to the `<30` class
because accuracy alone may conceal weak performance on this minority outcome.

## 6.2 Logistic Regression Training and Evaluation

This section keeps the original warm-start SAGA training idea while using the
shared, leakage-safe data preparation from Phases 1–5. The target remains encoded
as `NO=0`, `>30=1`, and `<30=2`. Model construction and the iterative
training loop are provided by `modules/logistic_regression_model.py`; experiment
presentation, diagnostics, evaluation, and artifact export remain in this
notebook.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
import joblib

from modules.logistic_regression_model import (
    train_logistic_regression_pipeline,
)

print("=" * 60)
print("LOGISTIC REGRESSION EXPERIMENT SETUP")
print("=" * 60)

lr_X_train = X_train
lr_X_test = X_test
lr_y_train = y_train
lr_y_test = y_test
LR_EARLY_READMISSION_LABEL = 2

print("Shared train shapes:", lr_X_train.shape, lr_y_train.shape)
print("Shared test shapes:", lr_X_test.shape, lr_y_test.shape)
print("Numerical feature count:", len(numerical_features))
print("Categorical feature count:", len(categorical_features))
print("Early-readmission label:", LR_EARLY_READMISSION_LABEL)
display(lr_y_train.value_counts().sort_index().to_frame("Count"))

### 6.2.1 Data Transformation and PCA Visualization

A two-component PCA projection is used only for visual inspection. It is not
inserted into the Logistic Regression model. The visualization preprocessor is
fit on the training set and only transforms the held-out test set.

In [ ]:
print("=" * 60)
print("LOGISTIC REGRESSION PCA VISUALIZATION")
print("=" * 60)

lr_visualization_preprocessor = get_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scaler_type="standard",
    log_numeric=False,
    min_category_frequency=20,
)
lr_X_train_transformed = lr_visualization_preprocessor.fit_transform(
    lr_X_train
)
lr_X_test_transformed = lr_visualization_preprocessor.transform(
    lr_X_test
)

lr_pca_visualization = PCA(
    n_components=2,
    random_state=RANDOM_STATE,
)
lr_X_train_pca = lr_pca_visualization.fit_transform(
    lr_X_train_transformed
)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    x=lr_X_train_pca[:, 0],
    y=lr_X_train_pca[:, 1],
    hue=lr_y_train.map(target_names),
    palette="viridis",
    alpha=0.5,
    s=15,
)
plt.title("2D PCA Visualization of Processed Training Data")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Readmitted Class")
plt.tight_layout()
plt.show()

del lr_X_train_transformed, lr_X_test_transformed, lr_X_train_pca

### 6.2.2 Warm-start Logistic Regression Training

The model retains the original optimization design: the SAGA solver performs one
iteration per loop with `warm_start=True`. Log loss is recorded after the first
iteration and every ten iterations through epoch 100.

In [ ]:
print("=" * 60)
print("LOGISTIC REGRESSION: OPTIMIZATION PROCESS")
print("=" * 60)
print("Solver: SAGA | Loss: Cross-Entropy | Max Iterations: 100")

(
    lr_pipeline,
    lr_loss_history,
    lr_training_seconds,
) = train_logistic_regression_pipeline(
    X_train=lr_X_train,
    y_train=lr_y_train,
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    epochs=100,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    min_category_frequency=20,
)

for row in lr_loss_history.itertuples(index=False):
    print(
        f"Iteration {row.iteration:4d} | "
        f"Cost (Log Loss): {row.log_loss:.4f}"
    )

print("-" * 60)
print(
    "Model training completed successfully in "
    f"{lr_training_seconds:.2f} seconds!"
)

lr_model = lr_pipeline.named_steps["model"]
LR_MODEL_PATH = MODELS_DIR / "logistic_regression_pipeline.joblib"
joblib.dump(lr_pipeline, LR_MODEL_PATH)

print("Full Logistic Regression pipeline saved to:")
print(LR_MODEL_PATH)

### 6.2.3 Sigmoid Decision Boundary Visualization

For the `<30` class, the plot compares its linear decision score with the
predicted one-vs-rest probability and the corresponding binary ground truth.

In [ ]:
print("=" * 60)
print("SIGMOID DECISION BOUNDARY VISUALIZATION")
print("=" * 60)

lr_test_transformed = lr_pipeline.named_steps[
    "preprocessor"
].transform(lr_X_test)
lr_class_index = list(lr_model.classes_).index(
    LR_EARLY_READMISSION_LABEL
)

lr_decision_scores = lr_model.decision_function(
    lr_test_transformed
)
lr_early_scores = (
    lr_decision_scores[:, lr_class_index]
    if lr_decision_scores.ndim > 1
    else lr_decision_scores
)
lr_early_probabilities = lr_model.predict_proba(
    lr_test_transformed
)[:, lr_class_index]

lr_sorted_index = np.argsort(lr_early_scores)
lr_y_test_binary = (
    lr_y_test == LR_EARLY_READMISSION_LABEL
).astype(int)

plt.figure(figsize=(10, 6))
plt.plot(
    lr_early_scores[lr_sorted_index],
    lr_early_probabilities[lr_sorted_index],
    color="red",
    linewidth=3,
    label="Sigmoid Curve (Predicted Probability)",
)
plt.scatter(
    lr_early_scores,
    lr_y_test_binary,
    alpha=0.1,
    color="blue",
    label="Actual Data ('<30' = 1, Other = 0)",
    s=15,
)
plt.title(
    "Logistic Regression: Sigmoid Function for '<30' Readmission Class",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Linear Score (z = W^T X)", fontsize=12)
plt.ylabel("Probability", fontsize=12)
plt.axhline(
    0.5,
    color="gray",
    linestyle="--",
    label="Decision Threshold (0.5)",
)
plt.legend(loc="center right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

del lr_test_transformed

### 6.2.4 Logistic Regression Evaluation

The held-out patient-disjoint test set is used for accuracy, multiclass
one-vs-rest ROC-AUC, the classification report, confusion matrix, and coefficient
interpretation.

In [ ]:
print("=" * 60)
print("LOGISTIC REGRESSION TEST PERFORMANCE")
print("=" * 60)

lr_y_pred = lr_pipeline.predict(lr_X_test)
lr_y_pred_proba = lr_pipeline.predict_proba(lr_X_test)

lr_accuracy = accuracy_score(
    lr_y_test,
    lr_y_pred,
)
lr_roc_auc = roc_auc_score(
    lr_y_test,
    lr_y_pred_proba,
    labels=LABELS,
    multi_class="ovr",
)

print(f"Overall Accuracy : {lr_accuracy:.4f}")
print(f"ROC AUC (OvR)    : {lr_roc_auc:.4f}")
print("\nDetailed Classification Report:")

lr_report_dict = classification_report(
    lr_y_test,
    lr_y_pred,
    labels=LABELS,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
print(
    classification_report(
        lr_y_test,
        lr_y_pred,
        labels=LABELS,
        target_names=CLASS_NAMES,
        zero_division=0,
    )
)

lr_report_df = pd.DataFrame(lr_report_dict).T
LR_REPORT_PATH = REPORTS_DIR / "logistic_regression_report.csv"
lr_report_df.to_csv(LR_REPORT_PATH, index=True)

print("Classification report exported to:")
print(LR_REPORT_PATH)

In [ ]:
print("=" * 60)
print("LOGISTIC REGRESSION CONFUSION MATRIX")
print("=" * 60)

lr_confusion_matrix = confusion_matrix(
    lr_y_test,
    lr_y_pred,
    labels=LABELS,
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    lr_confusion_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.title(
    "Confusion Matrix - Logistic Regression",
    fontsize=14,
    fontweight="bold",
)
plt.ylabel("Actual Label", fontsize=12)
plt.xlabel("Predicted Label", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("TOP 15 LOGISTIC FEATURES FOR '<30' READMISSION")
print("=" * 60)

lr_feature_names = lr_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()
lr_class_index = list(lr_model.classes_).index(
    LR_EARLY_READMISSION_LABEL
)
lr_coefficients = lr_model.coef_[lr_class_index]

lr_coefficient_df = pd.DataFrame(
    {
        "Feature": lr_feature_names,
        "Coefficient": lr_coefficients,
    }
)
lr_coefficient_df["Abs_Coefficient"] = lr_coefficient_df[
    "Coefficient"
].abs()
lr_top_features = (
    lr_coefficient_df
    .sort_values(by="Abs_Coefficient", ascending=False)
    .head(15)
)

display(lr_top_features.round(4))

plt.figure(figsize=(12, 8))
sns.barplot(
    data=lr_top_features,
    x="Coefficient",
    y="Feature",
    hue="Feature",
    palette="coolwarm",
    legend=False,
)
plt.title(
    "Top 15 Most Impactful Features for '<30' Readmission Risk",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel(
    "Logistic Regression Coefficient (Impact on Log-Odds)",
    fontsize=12,
)
plt.ylabel("Patient Feature", fontsize=12)
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

### Logistic Regression Experimental Summary

The implementation preserves the original warm-start SAGA optimization and
balanced class weighting while using the shared cleaned feature matrices. Its
coefficient analysis remains useful for identifying positive and negative
associations with early readmission. Because this merged notebook uses the
shared preprocessing path rather than the standalone notebook's raw-data path,
the numerical results are expected to differ from the standalone Logistic
Regression notebook.

## 6.3 Random Forest Training and Evaluation

This section retains the original nine Random Forest configurations and
patient-grouped cross-validation. Pipeline construction and the repeated
cross-validation loop are provided by
`modules/random_forest_model.py`; configuration declaration, model selection,
evaluation, visualization, and artifact export remain visible in the notebook.

In [ ]:
from sklearn.metrics import (
    auc,
    average_precision_score,
    balanced_accuracy_score,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import label_binarize

from modules.random_forest_model import (
    build_rf_pipeline,
    run_rf_cross_validation,
)

print("=" * 60)
print("RANDOM FOREST EXPERIMENT SETUP")
print("=" * 60)

rf_scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
}

print("Primary selection metric: Macro-F1")
print("Cross-validation:", group_cv)
print("Evaluation metrics:")
for metric in rf_scoring:
    print("-", metric)

In [ ]:
print("=" * 60)
print("RANDOM FOREST DUMMY CLASSIFIER BASELINE")
print("=" * 60)

rf_dummy_model = DummyClassifier(
    strategy="most_frequent"
)
rf_dummy_cv = cross_validate(
    estimator=rf_dummy_model,
    X=X_train,
    y=y_train,
    groups=groups_train,
    cv=group_cv,
    scoring=rf_scoring,
    n_jobs=2,
)
rf_dummy_summary = {
    metric: rf_dummy_cv[f"test_{metric}"].mean()
    for metric in rf_scoring
}

display(
    pd.DataFrame(
        [rf_dummy_summary],
        index=["Dummy - Most Frequent"],
    ).round(4)
)

In [ ]:
rf_configs = [
    {
        "name": "RF-1 | Standard | No PCA | Trees=100 | Depth=14 | Leaf=10 | No class weight",
        "scaler_type": "standard",
        "pca_variance": None,
        "n_estimators": 100,
        "max_depth": 14,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": None,
    },
    {
        "name": "RF-2 | Standard | No PCA | Trees=100 | Depth=14 | Leaf=10 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "n_estimators": 100,
        "max_depth": 14,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
    {
        "name": "RF-3 | Standard | No PCA | Trees=100 | Depth=14 | Leaf=10 | Balanced Subsample",
        "scaler_type": "standard",
        "pca_variance": None,
        "n_estimators": 100,
        "max_depth": 14,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": "balanced_subsample",
    },
    {
        "name": "RF-4 | MinMax | No PCA | Trees=100 | Depth=14 | Leaf=10 | Balanced",
        "scaler_type": "minmax",
        "feature_range": (0, 1),
        "pca_variance": None,
        "n_estimators": 100,
        "max_depth": 14,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
    {
        "name": "RF-5 | Standard | PCA 90% | Trees=100 | Depth=14 | Leaf=10 | Balanced",
        "scaler_type": "standard",
        "pca_variance": 0.90,
        "n_estimators": 100,
        "max_depth": 14,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
    {
        "name": "RF-6 | Standard | PCA 95% | Trees=100 | Depth=14 | Leaf=10 | Balanced",
        "scaler_type": "standard",
        "pca_variance": 0.95,
        "n_estimators": 100,
        "max_depth": 14,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
    {
        "name": "RF-7 | Standard | No PCA | Trees=100 | Depth=10 | Leaf=20 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_leaf": 20,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
    {
        "name": "RF-8 | Standard | No PCA | Trees=100 | Depth=18 | Leaf=5 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "n_estimators": 100,
        "max_depth": 18,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
    {
        "name": "RF-9 | Standard | No PCA | Trees=200 | Depth=14 | Leaf=10 | Balanced",
        "scaler_type": "standard",
        "pca_variance": None,
        "n_estimators": 200,
        "max_depth": 14,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
]

print(
    "Number of Random Forest configurations:",
    len(rf_configs),
)

In [ ]:
print("=" * 60)
print("RUN RANDOM FOREST CROSS-VALIDATION EXPERIMENTS")
print("=" * 60)

rf_results = run_rf_cross_validation(
    X_train=X_train,
    y_train=y_train,
    groups_train=groups_train,
    group_cv=group_cv,
    configs=rf_configs,
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scoring=rf_scoring,
    random_state=RANDOM_STATE,
    cv_n_jobs=1,
    model_n_jobs=-1,
    min_category_frequency=20,
)

display(rf_results.round(4))

In [ ]:
rf_ranking = rf_results[
    [
        "configuration",
        "cv_accuracy_mean",
        "cv_balanced_accuracy_mean",
        "cv_f1_macro_mean",
        "cv_f1_macro_std",
        "cv_f1_weighted_mean",
    ]
].copy()

display(rf_ranking.round(4))

rf_plot_df = rf_results.sort_values(
    "cv_f1_macro_mean",
    ascending=True,
)
rf_axis = rf_plot_df.plot(
    x="configuration",
    y="cv_f1_macro_mean",
    kind="barh",
    figsize=(11, 7),
    legend=False,
    color="#2b5c8f",
)
rf_axis.set_xlabel("Mean Macro-F1", fontsize=11)
rf_axis.set_ylabel("Configuration", fontsize=11)
rf_axis.set_title(
    "Random Forest Configuration Comparison - 5-Fold Grouped CV",
    fontsize=13,
    fontweight="bold",
)
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
rf_best_config_name = rf_results.iloc[0][
    "configuration"
]
rf_config_lookup = {
    config["name"]: config
    for config in rf_configs
}
rf_best_config = rf_config_lookup[
    rf_best_config_name
]

print("=" * 60)
print("BEST RANDOM FOREST CONFIGURATION")
print("=" * 60)
print("Configuration:", rf_best_config_name)
print("Scaler:", rf_best_config["scaler_type"])
print("PCA:", rf_best_config["pca_variance"])
print("Trees:", rf_best_config["n_estimators"])
print("Max Depth:", rf_best_config["max_depth"])
print(
    "Min Samples Leaf:",
    rf_best_config["min_samples_leaf"],
)
print(
    "Class Weight:",
    rf_best_config["class_weight"],
)
print(
    "Mean CV Macro-F1:",
    round(
        rf_results.iloc[0]["cv_f1_macro_mean"],
        4,
    ),
)
print(
    "Std CV Macro-F1:",
    round(
        rf_results.iloc[0]["cv_f1_macro_std"],
        4,
    ),
)

In [ ]:
print("=" * 60)
print("FIT BEST RANDOM FOREST ON FULL TRAINING SET")
print("=" * 60)

rf_best_model = build_rf_pipeline(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scaler_type=rf_best_config["scaler_type"],
    feature_range=rf_best_config.get(
        "feature_range",
        (0, 1),
    ),
    pca_variance=rf_best_config["pca_variance"],
    n_estimators=rf_best_config["n_estimators"],
    max_depth=rf_best_config["max_depth"],
    min_samples_leaf=rf_best_config[
        "min_samples_leaf"
    ],
    max_features=rf_best_config["max_features"],
    class_weight=rf_best_config["class_weight"],
    random_state=RANDOM_STATE,
    model_n_jobs=-1,
    min_category_frequency=20,
)
rf_best_model.fit(
    X_train,
    y_train,
)

print(
    "Best Random Forest fitted successfully "
    "on the full training set."
)

In [ ]:
print("=" * 60)
print("RANDOM FOREST HELD-OUT TEST EVALUATION")
print("=" * 60)

rf_y_test_pred = rf_best_model.predict(
    X_test
)
rf_y_test_probability = rf_best_model.predict_proba(
    X_test
)

rf_test_accuracy = accuracy_score(
    y_test,
    rf_y_test_pred,
)
rf_test_balanced_accuracy = balanced_accuracy_score(
    y_test,
    rf_y_test_pred,
)
rf_test_precision_macro = precision_score(
    y_test,
    rf_y_test_pred,
    average="macro",
    zero_division=0,
)
rf_test_recall_macro = recall_score(
    y_test,
    rf_y_test_pred,
    average="macro",
    zero_division=0,
)
rf_test_f1_macro = f1_score(
    y_test,
    rf_y_test_pred,
    average="macro",
    zero_division=0,
)
rf_test_f1_weighted = f1_score(
    y_test,
    rf_y_test_pred,
    average="weighted",
    zero_division=0,
)
rf_test_roc_auc_macro = roc_auc_score(
    y_test,
    rf_y_test_probability,
    labels=LABELS,
    multi_class="ovr",
    average="macro",
)

rf_test_summary = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Balanced Accuracy",
            "Macro Precision",
            "Macro Recall",
            "Macro F1",
            "Weighted F1",
            "Macro ROC-AUC (OvR)",
        ],
        "Score": [
            rf_test_accuracy,
            rf_test_balanced_accuracy,
            rf_test_precision_macro,
            rf_test_recall_macro,
            rf_test_f1_macro,
            rf_test_f1_weighted,
            rf_test_roc_auc_macro,
        ],
    }
)

display(rf_test_summary.round(4))

In [ ]:
print("=" * 60)
print("RANDOM FOREST CLASSIFICATION REPORT")
print("=" * 60)

rf_report = classification_report(
    y_test,
    rf_y_test_pred,
    labels=LABELS,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
rf_report_df = pd.DataFrame(rf_report).T
display(rf_report_df.round(4))

(
    rf_lt30_precision,
    rf_lt30_recall,
    rf_lt30_f1,
    rf_lt30_support,
) = precision_recall_fscore_support(
    y_test,
    rf_y_test_pred,
    labels=[2],
    zero_division=0,
)
rf_lt30_summary = pd.DataFrame(
    {
        "Metric": [
            "Precision",
            "Recall",
            "F1-score",
            "Support",
        ],
        "Value": [
            rf_lt30_precision[0],
            rf_lt30_recall[0],
            rf_lt30_f1[0],
            rf_lt30_support[0],
        ],
    }
)
display(rf_lt30_summary.round(4))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_y_test_pred,
    labels=LABELS,
    display_labels=CLASS_NAMES,
    normalize="true",
    cmap="Blues",
)
plt.title(
    "Normalized Random Forest Confusion Matrix - Test Set",
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("MULTICLASS ROC & PRECISION-RECALL CURVES (ONE-VS-REST)")
print("=" * 60)

rf_y_test_binary = label_binarize(
    y_test,
    classes=LABELS,
)
rf_macro_average_precision = average_precision_score(
    rf_y_test_binary,
    rf_y_test_probability,
    average="macro",
)
rf_lt30_average_precision = average_precision_score(
    (y_test == 2).astype(int),
    rf_y_test_probability[:, 2],
)

print(
    "Macro Average Precision (One-vs-Rest):",
    round(rf_macro_average_precision, 4),
)
print(
    "<30 Average Precision (One-vs-Rest):",
    round(rf_lt30_average_precision, 4),
)

rf_figure, rf_axes = plt.subplots(
    1,
    2,
    figsize=(16, 6),
)
rf_colors = ["#2b5c8f", "#d95f02", "#e7298a"]

for index, (color, class_name) in enumerate(
    zip(rf_colors, CLASS_NAMES)
):
    rf_fpr, rf_tpr, _ = roc_curve(
        rf_y_test_binary[:, index],
        rf_y_test_probability[:, index],
    )
    rf_class_auc = auc(rf_fpr, rf_tpr)
    rf_axes[0].plot(
        rf_fpr,
        rf_tpr,
        color=color,
        lw=2,
        label=(
            f"Class '{class_name}' "
            f"(AUC = {rf_class_auc:.3f})"
        ),
    )

rf_axes[0].plot(
    [0, 1],
    [0, 1],
    "k--",
    lw=1.2,
    label="Random (AUC = 0.500)",
)
rf_axes[0].set_xlabel("False Positive Rate")
rf_axes[0].set_ylabel("True Positive Rate")
rf_axes[0].set_title(
    f"ROC Curves OvR "
    f"(Macro AUC = {rf_test_roc_auc_macro:.3f})",
    fontweight="bold",
)
rf_axes[0].legend(loc="lower right")
rf_axes[0].grid(True, alpha=0.3)

for index, (color, class_name) in enumerate(
    zip(rf_colors, CLASS_NAMES)
):
    (
        rf_precision_curve,
        rf_recall_curve,
        _,
    ) = precision_recall_curve(
        rf_y_test_binary[:, index],
        rf_y_test_probability[:, index],
    )
    rf_pr_auc = average_precision_score(
        rf_y_test_binary[:, index],
        rf_y_test_probability[:, index],
    )
    rf_axes[1].plot(
        rf_recall_curve,
        rf_precision_curve,
        color=color,
        lw=2,
        label=(
            f"Class '{class_name}' "
            f"(AP = {rf_pr_auc:.3f})"
        ),
    )

rf_axes[1].set_xlabel("Recall")
rf_axes[1].set_ylabel("Precision")
rf_axes[1].set_title(
    f"Precision-Recall Curves "
    f"(Macro AP = {rf_macro_average_precision:.3f})",
    fontweight="bold",
)
rf_axes[1].legend(loc="upper right")
rf_axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
rf_final_result = pd.DataFrame(
    [
        {
            "Model": "Random Forest",
            "Best Configuration": (
                rf_best_config_name
            ),
            "Accuracy": rf_test_accuracy,
            "Balanced Accuracy": (
                rf_test_balanced_accuracy
            ),
            "Macro Precision": (
                rf_test_precision_macro
            ),
            "Macro Recall": rf_test_recall_macro,
            "Macro F1": rf_test_f1_macro,
            "Weighted F1": rf_test_f1_weighted,
            "<30 Precision": rf_lt30_precision[0],
            "<30 Recall": rf_lt30_recall[0],
            "<30 F1": rf_lt30_f1[0],
            "Macro ROC-AUC OvR": (
                rf_test_roc_auc_macro
            ),
            "Macro Average Precision OvR": (
                rf_macro_average_precision
            ),
        }
    ]
)

display(rf_final_result.round(4))

In [ ]:
print("=" * 60)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 60)

rf_preprocessor_fitted = rf_best_model.named_steps[
    "preprocessor"
]
rf_feature_names = np.asarray(
    rf_preprocessor_fitted.get_feature_names_out()
)
rf_importances = rf_best_model.named_steps[
    "rf"
].feature_importances_
rf_sorted_index = np.argsort(
    rf_importances
)[::-1]
RF_TOP_K = 20

rf_top_features_df = pd.DataFrame(
    {
        "Rank": np.arange(1, RF_TOP_K + 1),
        "Feature Name": rf_feature_names[
            rf_sorted_index[:RF_TOP_K]
        ],
        "Importance": rf_importances[
            rf_sorted_index[:RF_TOP_K]
        ],
        "Cumulative (%)": np.cumsum(
            rf_importances[
                rf_sorted_index[:RF_TOP_K]
            ]
        ) * 100,
    }
)

display(rf_top_features_df.round(4))

plt.figure(figsize=(11, 7))
sns.barplot(
    data=rf_top_features_df,
    x="Importance",
    y="Feature Name",
    hue="Feature Name",
    palette="Blues_r",
    legend=False,
)
plt.title(
    f"Top {RF_TOP_K} Most Important Features - Best Random Forest",
    fontsize=13,
    fontweight="bold",
)
plt.xlabel(
    "Gini Feature Importance (Mean Decrease in Impurity)"
)
plt.ylabel("Feature")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("EXPORT RANDOM FOREST RESULTS AND FEATURES")
print("=" * 60)

RF_REPORT_DIR = REPORTS_DIR / "rf"
RF_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

rf_results.to_csv(
    RF_REPORT_DIR / "rf_cv_results.csv",
    index=False,
)
rf_test_summary.to_csv(
    RF_REPORT_DIR / "rf_test_metrics.csv",
    index=False,
)
rf_final_result.to_csv(
    RF_REPORT_DIR / "rf_final_summary.csv",
    index=False,
)
rf_report_df.to_csv(
    RF_REPORT_DIR / "rf_classification_report.csv"
)
rf_top_features_df.to_csv(
    RF_REPORT_DIR / "rf_top_features.csv",
    index=False,
)

rf_export_preprocessor = get_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    scaler_type="standard",
    log_numeric=False,
    min_category_frequency=20,
)
rf_X_train_processed = (
    rf_export_preprocessor
    .fit_transform(X_train)
    .astype(np.float32)
)
rf_X_test_processed = (
    rf_export_preprocessor
    .transform(X_test)
    .astype(np.float32)
)

rf_pca_90_export = PCA(
    n_components=0.90,
    svd_solver="full",
    random_state=RANDOM_STATE,
)
rf_X_train_pca_90 = (
    rf_pca_90_export
    .fit_transform(rf_X_train_processed)
    .astype(np.float32)
)
rf_X_test_pca_90 = (
    rf_pca_90_export
    .transform(rf_X_test_processed)
    .astype(np.float32)
)

rf_pca_95_export = PCA(
    n_components=0.95,
    svd_solver="full",
    random_state=RANDOM_STATE,
)
rf_X_train_pca_95 = (
    rf_pca_95_export
    .fit_transform(rf_X_train_processed)
    .astype(np.float32)
)
rf_X_test_pca_95 = (
    rf_pca_95_export
    .transform(rf_X_test_processed)
    .astype(np.float32)
)

np.save(
    FEATURES_DIR / "X_train_processed.npy",
    rf_X_train_processed,
)
np.save(
    FEATURES_DIR / "X_test_processed.npy",
    rf_X_test_processed,
)
np.save(
    FEATURES_DIR / "X_train_pca90.npy",
    rf_X_train_pca_90,
)
np.save(
    FEATURES_DIR / "X_test_pca90.npy",
    rf_X_test_pca_90,
)
np.save(
    FEATURES_DIR / "X_train_pca95.npy",
    rf_X_train_pca_95,
)
np.save(
    FEATURES_DIR / "X_test_pca95.npy",
    rf_X_test_pca_95,
)
np.save(
    FEATURES_DIR / "y_train.npy",
    np.asarray(y_train, dtype=np.int64),
)
np.save(
    FEATURES_DIR / "y_test.npy",
    np.asarray(y_test, dtype=np.int64),
)

RF_MODEL_PATH = MODELS_DIR / "best_random_forest.joblib"
joblib.dump(
    rf_best_model,
    RF_MODEL_PATH,
)

print("Random Forest experiment results saved to:")
print(RF_REPORT_DIR)
print("Extracted feature matrices saved to:")
print(FEATURES_DIR)
print("Best Random Forest pipeline saved to:")
print(RF_MODEL_PATH)

### Random Forest Experimental Summary

The Random Forest section evaluates class weighting, scaling, optional PCA,
tree depth, minimum leaf size, and ensemble size using five-fold
patient-grouped cross-validation. Macro-F1 remains the primary selection metric,
and the held-out test set is evaluated only after the best configuration has
been selected. Feature importance is reported from the fitted champion model.